## Configurações

In [1]:
# Extensão que verifica se um dos arquivos foi alterado
%load_ext autoreload
%autoreload 2

In [2]:
from c3_1_novo_instance_reweighing import instance_reweighing  # É preciso começar por ele para não dar problema com o torch vindo da aif360

# Bibliotecas
import joblib
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
from sklearn.model_selection import train_test_split
import traceback

# Variáveis auxiliares
from c0_1_configuracoes import(
  param_grid_perceptron_basico,
  param_grid_random_forest_basico,
  param_grid_regressao_logistica_basico,
  param_grid_xgboost_basico,
  param_grid_perceptron_completo,
  param_grid_random_forest_completo,
  param_grid_regressao_logistica_completo,
  param_grid_xgboost_completo,
  preprocessor_passthrough,
  seed
)

from c0_2_cronometro import cronometro

# Funções auxiliares
from c1_6_enviesamento import enviesar

# Algoritmos
from c2_1_random_forest import random_forest_GSCV
from c2_2_xgboost import xgboost_GSCV
from c2_3_regressao_logistica import regressao_logistica_GSCV
from c2_4_perceptron import perceptron_GSCV

# Técnicas de pré-processamento
from c3_1_novo_instance_reweighing import instance_reweighing
from c3_2_novo_disparate_impact_removal import disparate_impact_removal
from c3_3_znovo_synthetic_data_generation import synthetic_data_generation
from c3_4_znovo_suppression import suppression

# Técnicas de pós-processamento
from c4_1_znovo_threshold_optimization import threshold_optimization
from c4_2_novo_calibration import calibration
from c4_3_znovo_reject_option_classification import reject_option_classification

# Interpretabilidade
from c5_1_interpretabilidade import gerar_interpretabilidade, gerar_interpretabilidade_especifica

# Gerar e interpretar resultados
from c6_1_znovo_salvar_resultados import gerar_planilha_nova as gerar_planilha, salvar_dicionario

# Garantir a replicabilidade
np.random.seed(seed)
random.seed(seed)

caminho_resultado = './Resultados'

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = vmap(

## Cálculo

In [3]:
# Chama a função que aplica todos os enviesamento considerando as variáveis sensiveis
# Não compensa salvar o arquivo pois seria muito pesado e leva apenas ~10 segundos para executar
datasets = enviesar()

In [4]:
# Removendo os originais dos datasets 1 e 2 | Removendo os smotes dos datasets 3 e 4

del datasets['df1']['original_sensitive_sexo']
del datasets['df1']['original_sensitive_age']
del datasets['df2']['original_sensitive_sexo']
del datasets['df2']['original_sensitive_age']
del datasets['df3']['smote_simples_sensitive_sexo']
del datasets['df3']['smote_simples_sensitive_age']
del datasets['df4']['smote_simples_sensitive_sexo']
del datasets['df4']['smote_simples_sensitive_age']

In [5]:
for df in datasets:
    print(df, datasets[df].keys())

df1 dict_keys(['smote_simples_sensitive_sexo', 'smote_simples_sensitive_age'])
df2 dict_keys(['smote_simples_sensitive_sexo', 'smote_simples_sensitive_age'])
df3 dict_keys(['original_sensitive_sexo', 'original_sensitive_age'])
df4 dict_keys(['original_sensitive_sexo', 'original_sensitive_age'])


In [ ]:
modo_completo = 1

param_grid_random_forest = param_grid_random_forest_completo if modo_completo else param_grid_random_forest_basico
param_grid_perceptron = param_grid_perceptron_completo if modo_completo else param_grid_perceptron_basico
param_grid_regressao_logistica = param_grid_regressao_logistica_completo if modo_completo else param_grid_regressao_logistica_basico
param_grid_xgboost = param_grid_xgboost_completo if modo_completo else param_grid_xgboost_basico

parametros_algoritmos = {
    'random_forest': {
        'funcao': random_forest_GSCV,
        'parametros_grid': param_grid_random_forest
    },
    'xgboost': {
        'funcao': xgboost_GSCV,
        'parametros_grid': param_grid_xgboost
    },
    'regressao_logistica': {
        'funcao': regressao_logistica_GSCV,
        'parametros_grid': param_grid_regressao_logistica
    },
    'perceptron': {
        'funcao': perceptron_GSCV,
        'parametros_grid': param_grid_perceptron
    }
}

printar_tecnicas = False

# Parâmetros comuns para todos
parametros_gerais = {
  'printar': True,
  'cv_n_splits': 2
}

resultado_global = {}
sucesso = 0

# Percorre cada base de dados distinta
try:

  for df in datasets:

    print(f"\n\n=== INICIANDO ANÁLISE DO {df.upper()} ===\n")
    # Percorre cada tipo de enviesamento
    for tipo in datasets[df]:

      print(f"\n--- Analisando o tipo: {tipo} ---\n")

      banco = datasets[df][tipo]
      nome_banco = banco['nome_banco']

      # Separando dados de treino e teste
      X_train = banco['treino'].drop('target',axis=1)
      y_train = banco['treino']['target']

      X_test = banco['teste'].drop('target', axis=1)
      y_test = banco['teste']['target']

      # Formando os parâmetros para as funções
      parametros_dataset = {
        'preprocessor': preprocessor_passthrough,
        'dados_sensiveis': banco['dados_sensiveis']
      }
      parametros_dataset.update(parametros_gerais)

      dados_dataset = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'nome_base_de_dados': nome_banco
      }

      resultado_dataset = {}
      resultado_dataset['smote'] = banco['smote']   # Tipo de smote usado no dataset em questão

      # Aplicando as técnicas de pré-processamento
      resultado_dataset['instance_reweighing']        = instance_reweighing         (parametros_dataset, dados_dataset, parametros_algoritmos, printar_tecnicas)
      resultado_dataset['disparate_impact_removal']   = disparate_impact_removal    (parametros_dataset, dados_dataset, parametros_algoritmos, printar_tecnicas)
      resultado_dataset['synthetic_data_generation']  = synthetic_data_generation   (parametros_dataset, dados_dataset, parametros_algoritmos, banco['colunas_discretas'],printar_tecnicas)
      resultado_dataset['suppression']                = suppression                 (parametros_dataset, dados_dataset, parametros_algoritmos, printar_tecnicas)

      # Aplicando as técnicas de pós-processamento + sem técnica
      resultado_dataset['threshold_optimization'] = {}
      resultado_dataset['calibration'] = {}
      resultado_dataset['reject_option_classification'] = {}
      resultado_dataset['sem_tecnica'] = {}

      # Separando os dados de treino (70%*) em treino (55%*) e validação (15%*) *Em relação ao total de dados
      X_train_pos, X_val_pos, y_train_pos, y_val_pos = train_test_split(X_train, y_train, test_size=15/70, random_state=seed, stratify=y_train)

      # Atualizando os dados com a nova divisão. Dados de teste e o nome da base de dados continuam iguais
      dados_dataset_pos = dados_dataset.copy()
      dados_dataset_pos['X_train'] = X_train_pos
      dados_dataset_pos['y_train'] = y_train_pos

      # Parametros para as técnicas de pós-processamento
      parametros_modelo_pos = parametros_dataset | dados_dataset_pos

      # Adicionando os dados de validação para as técnicas
      dados_dataset_pos['X_val'] = X_val_pos
      dados_dataset_pos['y_val'] = y_val_pos

      # Parâmetros para o modelo sem técnica
      parametros_modelo_sem_tecnica = parametros_dataset | dados_dataset
      parametros_modelo_sem_tecnica['nome_base_de_dados'] = parametros_modelo_sem_tecnica['nome_base_de_dados'] + " || SEM TÉCNICA"

      for nome_algoritmo in parametros_algoritmos:
        algoritmo = parametros_algoritmos[nome_algoritmo]

        # Modelo sem técnica
        parametros_modelo_sem_tecnica['param_grid'] = algoritmo['parametros_grid']
        _, resultado_dataset['sem_tecnica'][nome_algoritmo] = algoritmo['funcao'](**parametros_modelo_sem_tecnica)

        # Treinando o modelo base para as técnicas de pós-processamento
        parametros_modelo_pos['param_grid'] = algoritmo['parametros_grid']
        modelo, _ = algoritmo['funcao'](**parametros_modelo_pos)

        # Aplicando as técnicas de pós-processamento
        resultado_dataset['threshold_optimization'][nome_algoritmo]       = threshold_optimization        (parametros_dataset, dados_dataset_pos, modelo, nome_algoritmo)
        resultado_dataset['calibration'][nome_algoritmo]                  = calibration                   (parametros_dataset, dados_dataset_pos, modelo, nome_algoritmo)
        resultado_dataset['reject_option_classification'][nome_algoritmo] = reject_option_classification  (parametros_dataset, dados_dataset_pos, modelo, nome_algoritmo)

      resultado_global[f"{nome_banco.lower().replace(' ', '_')}"] = resultado_dataset.copy()

except KeyboardInterrupt:
  traceback.print_exc()

except Exception:
  traceback.print_exc()

else:
  sucesso = 1

finally:
  print("\n\n\n-------------------------------------------")
  salvar_dicionario(resultado_global, caminho_resultado, sucesso)
  gerar_planilha(resultado_global, caminho_resultado)
  print("-------------------------------------------\n\n\n")



=== INICIANDO ANÁLISE DO DF1 ===


--- Analisando o tipo: smote_simples_sensitive_sexo ---


----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || RANDOM FOREST || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.85      0.92      0.88      6805
           1       0.60      0.43      0.50      1958

    accuracy                           0.81      8763
   macro avg       0.72      0.67      0.69      8763
weighted avg       0.79      0.81      0.80      8763

ROC AUC: 0.7676
F1 Score: 0.5022
KS: 0.4071

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || RANDOM FOREST || MATRIZ DE CONFUSÃO -----

TN: 6234 | FP: 571
FN: 1110 | TP: 848

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || RANDOM FOREST || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1512
Selection Rate (Privilegiado):   0.1787
Demo

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || REGRESSAO LOGISTICA || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.85      0.92      0.88      6805
           1       0.59      0.42      0.49      1958

    accuracy                           0.80      8763
   macro avg       0.72      0.67      0.68      8763
weighted avg       0.79      0.80      0.79      8763

ROC AUC: 0.7133
F1 Score: 0.4879
KS: 0.3545

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || REGRESSAO LOGISTICA || MATRIZ DE CONFUSÃO -----

TN: 6231 | FP: 574
FN: 1141 | TP: 817

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || INSTANCE REWEIGHING || REGRESSAO LOGISTICA || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1518
Selection Rate (Privilegiado):   0.1696
Demographic Parity Difference: -0.0179
Demographic Parity Ratio:    0.8947

----

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || DISPARATE IMPACT REMOVAL || REGRESSAO LOGISTICA || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

         0.0       0.85      0.87      0.86      6805
         1.0       0.50      0.46      0.48      1958

    accuracy                           0.78      8763
   macro avg       0.67      0.66      0.67      8763
weighted avg       0.77      0.78      0.77      8763

ROC AUC: 0.6907
F1 Score: 0.4778
KS: 0.3306

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || DISPARATE IMPACT REMOVAL || REGRESSAO LOGISTICA || MATRIZ DE CONFUSÃO -----

TN: 5921 | FP: 884
FN: 1066 | TP: 892

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || DISPARATE IMPACT REMOVAL || REGRESSAO LOGISTICA || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1879
Selection Rate (Privilegiado):   0.2259
Demographic Parity Difference: -0.0381
Demographic Parity Ratio: 

...

ROC AUC: 0.8030
F1 Score: 0.5379
KS: 0.4791

----- DATASET 4 ORIGINAL COM SENSITIVE AGE || SUPPRESSION || REGRESSAO LOGISTICA || MATRIZ DE CONFUSÃO -----

TN: 181 | FP: 19
FN: 48 | TP: 39

----- DATASET 4 ORIGINAL COM SENSITIVE AGE || SUPPRESSION || REGRESSAO LOGISTICA || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.2213
Selection Rate (Privilegiado):   0.1879
Demographic Parity Difference: 0.0334
Demographic Parity Ratio:    1.1779

----- True Positive Rate (Equal Opportunity) -----
TPR (Desprivilegiado): 0.4000
TPR (Privilegiado):   0.5000
Diff: -0.1000 | Ratio: 0.8000

----- False Positive Rate -----
FPR (Desprivilegiado): 0.1169
FPR (Privilegiado):   0.0813
Diff: 0.0356 | Ratio: 1.4377

----- False Negative Rate -----
FNR Diff: 0.1000 | Ratio: 1.2000

----- Predictive Parity (Precision) -----
Precision (Desprivilegiado): 0.6667
Precision (Privilegiado):   0.6774
Diff: -0.0108 | Ratio: 0.9841

----- Equalized Odds -----

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:149: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.    0.    0.    0.    0.    1.    0.    1.    0.    0.    0.    0.
 0.008 0.    0.    0.008 0.    0.    1.    0.    0.    0.    0.    1.
 1.    0.    1.    0.    1.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    1.    1.    0.    0.    0.    0.    0.    0.    0.
 0.    1.    0.    0.008 0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.008 0.    0.    0.    1.    0.    0.
 0.    0.    1.    0.    0.    0.    0.    0.008 0.    1.    0.    0.
 0.    0.    1.    0.    0.    0.    0.    0.    0.    1.    0.    1.
 1.    0.    0.    1.    0.    0.    1.    1.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.008 1.    0.    0.008 1.    1.    0.
 1.    0.   ]' has dtype incompatible with float32, please 


----- DATASET 4 ORIGINAL COM SENSITIVE AGE || CALIBRATION || XGBOOST || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.81      0.83      0.82       200
           1       0.59      0.55      0.57        87

    accuracy                           0.75       287
   macro avg       0.70      0.69      0.70       287
weighted avg       0.74      0.75      0.75       287

ROC AUC: 0.7630
F1 Score: 0.5714
KS: 0.4432

----- DATASET 4 ORIGINAL COM SENSITIVE AGE || CALIBRATION || XGBOOST || MATRIZ DE CONFUSÃO -----

TN: 167 | FP: 33
FN: 39 | TP: 48

----- DATASET 4 ORIGINAL COM SENSITIVE AGE || CALIBRATION || XGBOOST || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.2623
Selection Rate (Privilegiado):   0.2970
Demographic Parity Difference: -0.0347
Demographic Parity Ratio:    0.8832

----- True Positive Rate (Equal Opportunity) -----
TPR (Desprivilegiado): 0.4889
TPR (P

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.2213
Selection Rate (Privilegiado):   0.1818
Demographic Parity Difference: 0.0395
Demographic Parity Ratio:    1.2172

----- True Positive Rate (Equal Opportunity) -----
TPR (Desprivilegiado): 0.4000
TPR (Privilegiado):   0.4762
Diff: -0.0762 | Ratio: 0.8400

----- False Positive Rate -----
FPR (Desprivilegiado): 0.1169
FPR (Privilegiado):   0.0813
Diff: 0.0356 | Ratio: 1.4377

----- False Negative Rate -----
FNR Diff: 0.0762 | Ratio: 1.1455

----- Predictive Parity (Precision) -----
Precision (Desprivilegiado): 0.6667
Precision (Privilegiado):   0.6667
Diff: 0.0000 | Ratio: 1.0000

----- Equalized Odds -----
Diff: 0.0762
Ratio: 0.8400


----- DATASET 4 ORIGINAL COM SENSITIVE AGE || SEM TÉCNICA || IMPORTÂNCIA DAS FEATURES -----

Attribute4_A41       1.672454 (Negativo)      
Attribute1_A14       1.533193 (Negativo)      
Attribute7_A74       0.964412 (Negativo)      
Attribute3_A34       0.891637 (Nega

## Controle manual

In [18]:
import os
import joblib

dicionario = './Resultados/resultado_global_dict_19_01_2026_17_27.joblib'

if os.path.isfile(dicionario):
  rf = joblib.load(dicionario)
  print("Dicionário recuperado")

In [19]:
# Salvar resultado manualmente
gerar_planilha(rf, caminho_resultado)

Arquivo Excel salvo em ./Resultados/resultado_global_09_02_2026_02_27.xlsx


In [ ]:
vencedores = [
    # Dataset Original
    {'ds': 'dataset_1_original_com_sensitive_age', 'mo': 'xgboost', 'te': 'threshold_optimization'},
    {'ds': 'dataset_1_original_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'synthetic_data_generation'},
    {'ds': 'dataset_2_original_com_sensitive_age', 'mo': 'regressão_logística', 'te': 'reject_option_classification'},
    {'ds': 'dataset_2_original_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'synthetic_data_generation'},
    {'ds': 'dataset_3_original_com_sensitive_age', 'mo': 'random_forest', 'te': 'calibration'},
    {'ds': 'dataset_3_original_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'sem_tecnica'},
    {'ds': 'dataset_4_original_com_sensitive_age', 'mo': 'xgboost', 'te': 'sem_tecnica'},
    {'ds': 'dataset_4_original_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'sem_tecnica'},
    
    # Algoritmos Vencedores:
      # XGBoost: 6 vezes
      # Random Forest: 1 vez
      # Regressão Logística: 1 vez
    # Técnicas Vencedoras:
      # sem_tecnica: 3 vezes (D3-Sexo, D4-Idade, D4-Sexo)
      # Synthetic Data Generation: 2 vezes (D1-Sexo, D2-Sexo)
      # Threshold Optimization: 1 vez (D1-Idade)
      # Reject Option Classification: 1 vez (D2-Idade)
      # Calibration: 1 vez (D3-Idade)

    # Dataset SMOTE
    {'ds': 'dataset_1_smote_simples_com_sensitive_age', 'mo': 'random_forest', 'te': 'suppression'},
    {'ds': 'dataset_1_smote_simples_com_sensitive_sexo', 'mo': 'random_forest', 'te': 'synthetic_data_generation'},
    {'ds': 'dataset_2_smote_simples_com_sensitive_age', 'mo': 'xgboost', 'te': 'instance_reweighing'},
    {'ds': 'dataset_2_smote_simples_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'instance_reweighing'},
    {'ds': 'dataset_3_smote_simples_com_sensitive_age', 'mo': 'xgboost', 'te': 'suppression'},
    {'ds': 'dataset_3_smote_simples_com_sensitive_sexo', 'mo': 'xgboost', 'te': 'synthetic_data_generation'},
    {'ds': 'dataset_4_smote_simples_com_sensitive_age', 'mo': 'regressão_logística', 'te': 'disparate_impact_removal'},
    {'ds': 'dataset_4_smote_simples_com_sensitive_sexo', 'mo': 'regressão_logística', 'te': 'threshold_optimization'},

    # Algoritmos Vencedores:
      # XGBoost: 4 vezes
      # Random Forest: 2 vezes
      # Regressão Logística: 2 vezes
    # Técnicas Vencedoras:
      # Instance Reweighing: 2 vezes (D2-Idade, D2-Sexo)
      # Suppression: 2 vezes (D1-Idade, D3-Idade)
      # Synthetic Data Generation: 2 vezes (D1-Sexo, D3-Sexo)
      # Disparate Impact Removal: 1 vez (D4-Idade)
      # Threshold Optimization: 1 vez (D4-Sexo)

]

def plotar_importancia(dados_lista, titulo):
    if not dados_lista:
        return
    
    df = pd.DataFrame(dados_lista)
    df = df.sort_values(by='importancia', ascending=True)
    
    altura_dinamica = max(6, len(df) * 0.2) 
    plt.figure(figsize=(10, altura_dinamica))
    
    cores = [
        '#e67e22' if 'sensitive' in str(var).lower() else '#4c72b0' 
        for var in df['variavel']
    ]
    
    barras = plt.barh(df['variavel'], df['importancia'], color=cores, height=0.7)
    
    plt.xlabel('Importância', fontsize=12)
    plt.ylabel('Variáveis', fontsize=12)
    plt.title(titulo, fontsize=14, pad=20)
    
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

historico = {}

for v in vencedores:
    print(f"\n{'='*80}")
    print(f"PROCESSANDO: {v['ds']}")
    print(f"MODELO: {v['mo']} | TÉCNICA: {v['te']}")
    print(f"{'='*80}")
    
    try:
        resultado = gerar_interpretabilidade_especifica(rf, v['ds'], v['mo'], v['te'])
        historico[v['ds']] = resultado

        if resultado['shap']:
            plotar_importancia(resultado['shap'], f"SHAP - {v['mo']} ({v['te']})\n{v['ds']}")
        
        if resultado['permutation']:
            plotar_importancia(resultado['permutation'], f"Permutation Importance - {v['mo']} ({v['te']})\n{v['ds']}")

        if resultado['lime']:
            plotar_importancia(resultado['lime'], f"LIME - {v['mo']} ({v['te']})\n{v['ds']}")

    except Exception as e:
        print(f"Erro ao gerar interpretabilidade para {v['ds']}: {e}")

In [1]:
# Salvar dicionário de interpretabilidade
joblib.dump(historico, './Resultados/interpretabilidade.joblib')
recuperado = joblib.load('./Resultados/interpretabilidade.joblib')

NameError: name 'joblib' is not defined

In [ ]:
# gerar_interpretabilidade(rf, '.')     # Gera a interpretabilidade de todos os testes do dicionário de resultados

In [5]:
# Para visualizar as chaves do dicionário de resultados

def listar_hierarquia_ordenada(dicionario, nivel=0):
    # Ordenar as chaves
    for chave, valor in sorted(dicionario.items()):
        indentacao = "    " * nivel
        print(f"{indentacao}{chave}")
        
        # Acessa os demais dicionários recursivamente
        if isinstance(valor, dict):
            listar_hierarquia_ordenada(valor, nivel + 1)

listar_hierarquia_ordenada(rf)

dataset_1_original_com_sensitive_age
    calibration
        perceptron
            desprivilegiado
                F1_Score
                KS
                ROC_AUC
                matriz_de_confusao
                relatorio_classificacao
                    0
                        f1-score
                        precision
                        recall
                        support
                    1
                        f1-score
                        precision
                        recall
                        support
                    accuracy
                    macro avg
                        f1-score
                        precision
                        recall
                        support
                    weighted avg
                        f1-score
                        precision
                        recall
                        support
            geral
                F1_Score
                KS
                ROC_AUC
               

In [19]:
from c6_1_znovo_salvar_resultados import gerar_planilha_nova

gerar_planilha_nova(rf, '.')

Arquivo Excel salvo em ./resultado_global_19_01_2026_17_39.xlsx
